# 🚦 Kochi Smart Traffic — ML Training Notebook v2
**Models:** XGBoost for Congestion Level + Accident Risk Level  
**Dataset:** Real TomTom + OpenWeatherMap schema (or synthetic equivalent)  
**Features:** 19 features including rainfall, visibility, temperature, free-flow speed

---


## Cell 1 — Install Dependencies

In [ ]:
!pip install xgboost scikit-learn imbalanced-learn pandas numpy matplotlib seaborn --quiet

## Cell 2 — Upload Dataset
Run this cell and use the file picker to upload your CSV file.  
Works with both **kochi_traffic_synthetic_v2.csv** and the real collected **kochi_traffic_real.csv**

In [ ]:
from google.colab import files
import pandas as pd
import io

print('Please upload your CSV file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Loaded: {filename}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## Cell 3 — Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Kochi Traffic Dataset — Overview', fontsize=14, fontweight='bold')

# Congestion level distribution
order = ['Low', 'Moderate', 'High', 'Very High']
colors = ['#00c853', '#ffd600', '#ff6d00', '#d50000']
vc = df['congestion_level'].value_counts().reindex(order)
axes[0,0].bar(vc.index, vc.values, color=colors)
axes[0,0].set_title('Congestion Level Distribution')
axes[0,0].set_xlabel('Level')

# Weather distribution
df['weather_condition'].value_counts().plot(kind='bar', ax=axes[0,1], color='#1565c0')
axes[0,1].set_title('Weather Condition Distribution')
axes[0,1].tick_params(axis='x', rotation=30)

# Congestion by hour
hourly = df.groupby('hour')['congestion_score'].mean()
axes[0,2].plot(hourly.index, hourly.values, color='#e53935', linewidth=2, marker='o', markersize=4)
axes[0,2].set_title('Avg Congestion Score by Hour')
axes[0,2].set_xlabel('Hour of Day')
axes[0,2].axvspan(7, 9, alpha=0.15, color='red', label='Rush')
axes[0,2].axvspan(17, 20, alpha=0.15, color='red')
axes[0,2].legend()

# Congestion by weather
df.groupby('weather_condition')['congestion_score'].mean().sort_values().plot(
    kind='barh', ax=axes[1,0], color='#0288d1')
axes[1,0].set_title('Avg Congestion by Weather')

# Temperature distribution
axes[1,1].hist(df['temperature_c'], bins=30, color='#f57c00', edgecolor='white')
axes[1,1].set_title('Temperature Distribution (°C)')

# Rainfall vs congestion
sample = df.sample(min(2000, len(df)))
axes[1,2].scatter(sample['rainfall_mm'], sample['congestion_score'],
                  alpha=0.3, color='#6a1b9a', s=10)
axes[1,2].set_title('Rainfall vs Congestion Score')
axes[1,2].set_xlabel('Rainfall (mm)')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA complete')

## Cell 4 — Preprocessing & Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pickle

# Drop rows with nulls in key columns
required = ['congestion_score', 'current_speed_kmph', 'free_flow_speed_kmph',
            'rainfall_mm', 'visibility_m', 'temperature_c', 'humidity_pct']
df = df.dropna(subset=required)
print(f'Rows after cleaning: {len(df):,}')

# Cyclic time encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_sin']  = np.sin(2 * np.pi * df['day_of_week'].map(
    {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
     'Friday':4,'Saturday':5,'Sunday':6}) / 7)
df['day_cos']  = np.cos(2 * np.pi * df['day_of_week'].map(
    {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
     'Friday':4,'Saturday':5,'Sunday':6}) / 7)

# Speed ratio (how much below free-flow)
df['speed_ratio'] = (df['current_speed_kmph'] / df['free_flow_speed_kmph'].replace(0, 1)).clip(0, 1.5)

# Label encode categoricals
label_encoders = {}
cat_cols = ['area', 'road_name', 'road_type', 'weather_condition', 'day_of_week',
            'congestion_level', 'accident_risk_level']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

print('Label encoders saved')
print(f'Feature engineering complete. Shape: {df.shape}')

## Cell 5 — Define Features

In [ ]:
FEATURES = [
    # Time
    'hour', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'is_rush_hour', 'is_weekend', 'is_night',
    # Road
    'lanes', 'speed_limit_kmph', 'road_type_enc',
    # Traffic
    'current_speed_kmph', 'free_flow_speed_kmph', 'speed_ratio',
    'vehicle_volume_est', 'congestion_score',
    # Weather
    'weather_condition_enc', 'rainfall_mm', 'visibility_m',
    'temperature_c', 'humidity_pct',
    # Location
    'area_enc', 'road_name_enc',
]

import pickle
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

print(f'Using {len(FEATURES)} features:')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:2d}. {f}')

## Cell 6 — Train Congestion Level Model

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import pickle

X = df[FEATURES]
y = df['congestion_level_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

cong_model = XGBClassifier(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
cong_model.fit(X_train, y_train,
               eval_set=[(X_test, y_test)],
               verbose=50)

y_pred = cong_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\nAccuracy: {acc:.4f} ({acc*100:.1f}%)')

le_cong = label_encoders['congestion_level']
present = sorted(y_test.unique())
labels  = le_cong.inverse_transform(present)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, labels=present, target_names=labels))

fig, ax = plt.subplots(figsize=(7,5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=labels, ax=ax)
ax.set_title('Congestion Model — Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_congestion.png', dpi=120)
plt.show()

with open('congestion_model.pkl', 'wb') as f:
    pickle.dump(cong_model, f)
print('congestion_model.pkl saved')

## Cell 7 — Train Accident Risk Model

In [ ]:
from imblearn.over_sampling import SMOTE
import pickle

RISK_FEATURES = FEATURES + ['congestion_score', 'congestion_level_enc']
RISK_FEATURES = list(dict.fromkeys(RISK_FEATURES))  # deduplicate

Xr = df[RISK_FEATURES]
yr = df['accident_risk_level_enc']

# Drop classes with fewer than 2 samples
valid_classes = yr.value_counts()[yr.value_counts() >= 2].index
mask = yr.isin(valid_classes)
Xr, yr = Xr[mask], yr[mask]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42)

# SMOTE to balance classes
k = min(3, yr_train.value_counts().min() - 1)
if k >= 1:
    sm = SMOTE(random_state=42, k_neighbors=k)
    Xr_train, yr_train = sm.fit_resample(Xr_train, yr_train)
    print(f'After SMOTE: {len(Xr_train):,} training samples')

risk_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
risk_model.fit(Xr_train, yr_train,
               eval_set=[(Xr_test, yr_test)],
               verbose=50)

yr_pred = risk_model.predict(Xr_test)
acc_r = accuracy_score(yr_test, yr_pred)
print(f'\nAccuracy: {acc_r:.4f} ({acc_r*100:.1f}%)')

le_risk = label_encoders['accident_risk_level']
present_r = sorted(yr_test.unique())
labels_r  = le_risk.inverse_transform(present_r)
print('\nClassification Report:')
print(classification_report(yr_test, yr_pred, labels=present_r, target_names=labels_r))

fig, ax = plt.subplots(figsize=(7,5))
ConfusionMatrixDisplay.from_predictions(yr_test, yr_pred, display_labels=labels_r, ax=ax)
ax.set_title('Risk Model — Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_risk.png', dpi=120)
plt.show()

with open('accident_risk_model.pkl', 'wb') as f:
    pickle.dump(risk_model, f)

# Save risk feature list separately
with open('risk_feature_columns.pkl', 'wb') as f:
    pickle.dump(RISK_FEATURES, f)

print('accident_risk_model.pkl saved')

## Cell 8 — Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model, feats, title in [
    (axes[0], cong_model, FEATURES, 'Congestion Model'),
    (axes[1], risk_model, RISK_FEATURES, 'Risk Model')
]:
    importance = pd.Series(model.feature_importances_, index=feats)
    importance.nlargest(15).sort_values().plot(kind='barh', ax=ax, color='#1b5e20')
    ax.set_title(f'{title} — Top 15 Features')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

## Cell 9 — Save Class Orders & Download All

In [ ]:
import pickle

congestion_order = ['Low', 'Moderate', 'High', 'Very High']
risk_order       = ['Low', 'Moderate', 'High', 'Very High']

with open('class_orders.pkl', 'wb') as f:
    pickle.dump({'congestion_order': congestion_order, 'risk_order': risk_order}, f)

print('All model files ready:')
import os
for fname in ['congestion_model.pkl', 'accident_risk_model.pkl',
              'label_encoders.pkl', 'feature_columns.pkl',
              'risk_feature_columns.pkl', 'class_orders.pkl']:
    size = os.path.getsize(fname) / 1024
    print(f'  {fname:<35} {size:.1f} KB')

## Cell 10 — Zip & Download Models

In [ ]:
import zipfile, os
from google.colab import files

model_files = [
    'congestion_model.pkl',
    'accident_risk_model.pkl',
    'label_encoders.pkl',
    'feature_columns.pkl',
    'risk_feature_columns.pkl',
    'class_orders.pkl',
]

with zipfile.ZipFile('kochi_traffic_models_v2.zip', 'w') as zf:
    for fname in model_files:
        zf.write(fname)

files.download('kochi_traffic_models_v2.zip')
print('Download started — extract to your project models/ folder')